# 01 - Benchmark construction and label verification

Verifies that all 360 samples exist and that every label matches the SARD ground truth. No API calls.

In [5]:
# --- environment ---
!pip -q install openai
from google.colab import drive; drive.mount('/content/drive')

import sys
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/LLM_Security_Paper')   # SARD corpus root
WORK = ROOT / 'revision_2026'                              # outputs live here
WORK.mkdir(exist_ok=True)

sys.path.insert(0, str(WORK))          # vulnbench.py lives in WORK
import vulnbench as vb

BENCH = pd.read_csv(WORK / 'benchmark_360_metadata.csv')
print(len(BENCH), 'samples |', BENCH.true_label.value_counts().to_dict())

Mounted at /content/drive
360 samples | {'Safe': 180, 'Vulnerable': 180}


In [6]:
# --- verify every file exists and every label matches SARD ground truth ---
import re
missing, mismatched = [], []
for _, r in BENCH.iterrows():
    p = ROOT / r['rel_path']
    if not p.exists():
        missing.append(r['rel_path'])
        continue
    head = p.read_text(encoding='utf-8', errors='replace')[:2000]
    truth = 'Vulnerable' if re.search(r'\bUnsafe sample\b', head, re.I) else 'Safe'
    if truth != r['true_label']:
        mismatched.append((r['rel_path'], r['true_label'], truth))

print('missing   :', len(missing))
print('mismatched:', len(mismatched))
assert not missing and not mismatched, 'benchmark integrity check failed'
print('OK - all 360 files present, all labels match SARD ground truth')

missing   : 0
mismatched: 0
OK - all 360 files present, all labels match SARD ground truth


In [7]:
# --- composition report (for the paper) ---
print(pd.crosstab(BENCH.cwe, BENCH.true_label))
print()
print('structurally unique :', BENCH.body_hash.nunique(), '/', len(BENCH))
print('distinct sanitizers :', BENCH.sanitizer.nunique())
print('distinct sources    :', BENCH.source.nunique())
print('distinct sinks      :', BENCH.sink.nunique())

true_label  Safe  Vulnerable
cwe                         
CWE_78        60          60
CWE_89        60          60
CWE_98        60          60

structurally unique : 360 / 360
distinct sanitizers : 35
distinct sources    : 16
distinct sinks      : 46
